<a href="https://colab.research.google.com/github/jhhlim/LLMFundamentals/blob/main/Jason_Lim_hw_3a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Class 3a — LLM Fundamentals (UCSC Extension)
# Homework 3a: Sentiment Analysis with VADER

**Student:** Jason Lim  
**Submit to:** svagarwa@ucsc.edu  
**Filename:** `Jason_Lim_hw_3a.ipynb`

## Goal
Apply **VADER** (Valence Aware Dictionary and sEntiment Reasoner) sentiment analysis to **unstructured free-text product reviews**, then compare VADER predictions against the dataset’s ground-truth labels.

## Dataset
**UCI Sentiment Labelled Sentences — Amazon cell phone / product reviews**  
https://archive.ics.uci.edu/dataset/331/sentiment+labelled+sentences

- 1,000 Amazon product review sentences
- Label `1` = positive, `0` = negative
- Short, clearly polar review sentences (good homework-sized unstructured text)

## References
- Class VADER notebook: Valence Aware Dictionary and sEntiment Reasoner
- Class 3a / Hands-On LLM Ch. 4 themes: text classification + evaluation with `classification_report`


## Step 0 — Install + imports

In [ ]:
%%capture
!pip install -q vaderSentiment pandas matplotlib seaborn scikit-learn


In [ ]:
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sns.set_theme(style="whitegrid")
%matplotlib inline


## Step 1 — Load Amazon product reviews (UCI)

Format of each line: `sentence <TAB> score` where score is 0 (negative) or 1 (positive).


In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip"

with urlopen(DATA_URL) as resp:
    zf = ZipFile(BytesIO(resp.read()))

# Product reviews: Amazon cell-phone accessories / electronics sentences
with zf.open("sentiment labelled sentences/amazon_cells_labelled.txt") as f:
    df = pd.read_csv(f, sep="\t", header=None, names=["text", "label"])

print("Shape:", df.shape)
print("Label counts:\n", df["label"].value_counts().sort_index())
df.head(10)


## Step 2 — Quick look at positive vs negative reviews

Unstructured free-text examples from the product review corpus.


In [ ]:
print("=== Positive examples (label=1) ===")
for t in df.loc[df["label"] == 1, "text"].head(5):
    print("-", t)

print("\n=== Negative examples (label=0) ===")
for t in df.loc[df["label"] == 0, "text"].head(5):
    print("-", t)


## Step 3 — VADER warm-up (same idea as the class VADER notebook)

VADER returns `neg`, `neu`, `pos`, and a **compound** score in `[-1, 1]`.
The compound score is the main overall tone signal.


In [ ]:
analyzer = SentimentIntensityAnalyzer()

angry_review = "The food was disgusting. I am never coming back here again!!"
great_review = "This phone case is excellent value. I am going to buy another one!!!"

print("Angry review scores:", analyzer.polarity_scores(angry_review))
print("Great review scores:", analyzer.polarity_scores(great_review))


In [ ]:
def classify_text(text, threshold=0.0):
    """Classify text with VADER using the compound score.

    threshold=0 follows the class VADER notebook:
      compound >= threshold → positive (1)
      compound <  threshold → negative (0)
    """
    scores = analyzer.polarity_scores(text)
    compound = scores["compound"]
    pred_class = 1 if compound >= threshold else 0
    return pred_class, compound, scores


for example in [
    great_review,
    "the food was really bad",
    "what is your name",
    "Great battery life but the mic is awful.",
]:
    pred, compound, scores = classify_text(example)
    print(f"pred={pred}  compound={compound:+.3f}  |  {example}")
    print(" ", scores)


## Step 4 — Run VADER on all Amazon product reviews

Store compound / pos / neu / neg scores and a binary prediction.


In [ ]:
rows = []
for text in df["text"]:
    pred, compound, scores = classify_text(text, threshold=0.0)
    rows.append(
        {
            "pred": pred,
            "compound": compound,
            "vader_pos": scores["pos"],
            "vader_neu": scores["neu"],
            "vader_neg": scores["neg"],
        }
    )

scored = pd.concat([df.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
scored["correct"] = scored["pred"] == scored["label"]
scored.head(10)


## Step 5 — Evaluate against ground-truth labels

In [ ]:
y_true = scored["label"]
y_pred = scored["pred"]

print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print()
print(
    classification_report(
        y_true, y_pred, target_names=["Negative Review", "Positive Review"]
    )
)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred Neg", "Pred Pos"],
    yticklabels=["True Neg", "True Pos"],
    ax=ax,
)
ax.set_title("VADER on Amazon product reviews")
plt.tight_layout()
plt.show()


## Step 6 — Visualize the compound-score distribution

Near-zero compound scores are more neutral / mixed. Strong positives and negatives sit near ±1.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(
    data=scored,
    x="compound",
    hue="label",
    bins=30,
    ax=axes[0],
    palette=["#d62728", "#2ca02c"],
)
axes[0].axvline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("VADER compound by true label (0=neg, 1=pos)")

sns.boxplot(
    data=scored,
    x="label",
    y="compound",
    ax=axes[1],
    palette=["#d62728", "#2ca02c"],
)
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Compound score by true label")
axes[1].set_xticklabels(["Negative", "Positive"])

plt.tight_layout()
plt.show()

print("Mean compound by true label:")
print(scored.groupby("label")["compound"].mean())


## Step 7 — Error analysis

Look at a few misclassified reviews. These often contain mixed sentiment, sarcasm, or domain words VADER underweights.


In [ ]:
errors = scored.loc[~scored["correct"]].copy()
print(f"Misclassified: {len(errors)} / {len(scored)} ({100 * len(errors) / len(scored):.1f}%)\n")

print("=== False negatives (true=1, pred=0): positive reviews VADER called negative ===")
fn = errors[errors["label"] == 1].sort_values("compound").head(8)
for _, r in fn.iterrows():
    print(f"compound={r['compound']:+.3f} | {r['text']}")

print("\n=== False positives (true=0, pred=1): negative reviews VADER called positive ===")
fp = errors[errors["label"] == 0].sort_values("compound", ascending=False).head(8)
for _, r in fp.iterrows():
    print(f"compound={r['compound']:+.3f} | {r['text']}")


## Step 8 — Optional threshold experiment

The class notebook used `threshold=0`. Trying a small dead-zone around 0 can treat near-neutral reviews as a separate bucket for business filtering (as noted in the VADER lab).


In [ ]:
def ternary_label(compound, pos_thresh=0.05, neg_thresh=-0.05):
    if compound >= pos_thresh:
        return "positive"
    if compound <= neg_thresh:
        return "negative"
    return "neutral / mixed"

scored["vader_ternary"] = scored["compound"].apply(ternary_label)
print(scored["vader_ternary"].value_counts())
print()

# Among clearly polar VADER predictions, check agreement with ground truth
polar = scored[scored["vader_ternary"] != "neutral / mixed"].copy()
polar_pred = (polar["vader_ternary"] == "positive").astype(int)
print(
    "Accuracy on non-neutral VADER calls only:",
    round(accuracy_score(polar["label"], polar_pred), 4),
)
print("Count used:", len(polar), "/", len(scored))


## Learnings / analysis notes

1. **Dataset choice:** Amazon cell-product review sentences are short, unstructured free text with clear polarity — a good fit for VADER homework.
2. **VADER compound score:** Combines lexicon valence with punctuation / capitalization / intensifiers. `>= 0` → positive class in the class notebook convention.
3. **Evaluation:** Comparing against UCI labels with `classification_report` + confusion matrix connects the VADER lab to Class 3a text-classification evaluation practice.
4. **Errors:** Mixed reviews (“great battery but awful mic”) and domain jargon can confuse lexicon methods. Near-zero compound scores are useful to filter out low-signal text before business analytics.
5. **Next step (optional):** Swap VADER for a transformer sentiment pipeline (Class 3a / Ch. 4) and compare accuracy on the same Amazon reviews.
